# Project 4 — Mobile Connectivity Improvement
## Independent Business Analyst Portfolio Project

**Purpose:** Build a reproducible, evidence-based analysis of mobile/internet connectivity gaps using public World Bank/ITU data, then translate the findings into a BA prioritisation framework and prototype decision process.

### Important portfolio disclosure
This is an **independent case study**. Public data are real; any GSMA internal workflow, stakeholder interviews, targets, weights, requirements, and process assumptions are proposed/simulated and must be labelled as such in the final portfolio.

### Learning mode
The notebook is intentionally structured so a beginner can follow:
1. Data acquisition
2. Data quality audit
3. Cleaning
4. Exploratory analysis
5. Connectivity gap / population impact
6. Root-cause exploration
7. Priority scoring
8. Visualisations
9. BA requirements and process outputs
10. Export of portfolio-ready tables


In [ ]:
# 1. SETUP
# Run this cell first.

import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Environment ready.")


## 2. Configuration

We use the World Bank API rather than manually downloading files. This makes the analysis easier to reproduce.

### Indicators used

- `IT.NET.USER.ZS` — Individuals using the Internet (% of population)
- `IT.CEL.SETS.P2` — Mobile cellular subscriptions (per 100 people)
- `SP.POP.TOTL` — Population, total
- `NY.GDP.PCAP.CD` — GDP per capita (current US$)
- `SP.RUR.TOTL.ZS` — Rural population (% of total population)

**Important:** These indicators do not measure the same thing. In particular, mobile subscriptions are not equivalent to unique mobile Internet users.


In [ ]:
# 2. CONFIGURATION

BASE_URL = "https://api.worldbank.org/v2"

INDICATORS = {
    "internet_usage_pct": "IT.NET.USER.ZS",
    "mobile_subscriptions_per_100": "IT.CEL.SETS.P2",
    "population": "SP.POP.TOTL",
    "gdp_per_capita": "NY.GDP.PCAP.CD",
    "rural_population_pct": "SP.RUR.TOTL.ZS",
}

# We initially retrieve a broad recent window and later select the latest
# comparable observation available for each country/indicator.
START_YEAR = 2015
END_YEAR = 2025

print("Indicators configured:", len(INDICATORS))


## 3. Reusable World Bank API function

The function below:
- sends a request to the World Bank API;
- checks whether the request succeeded;
- handles pagination;
- converts the API response into a pandas DataFrame.

This is better than hard-coding downloaded numbers into the notebook.


In [ ]:
# 3. WORLD BANK API FUNCTION

def fetch_world_bank_indicator(indicator_code, start_year=2015, end_year=2025, country="all"):
    url = (
        f"{BASE_URL}/country/{country}/indicator/{indicator_code}"
        f"?format=json&per_page=20000&date={start_year}:{end_year}"
    )

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    payload = response.json()

    if not isinstance(payload, list) or len(payload) < 2:
        raise ValueError(f"Unexpected API response for {indicator_code}")

    metadata, records = payload[0], payload[1]

    df = pd.DataFrame(records)

    if df.empty:
        return df

    # Keep the fields most useful for this project.
    keep = [
        "countryiso3code",
        "country",
        "date",
        "value",
        "indicator"
    ]
    keep = [c for c in keep if c in df.columns]
    df = df[keep].copy()

    df["date"] = pd.to_numeric(df["date"], errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    return df


# Test one indicator
test_df = fetch_world_bank_indicator(INDICATORS["internet_usage_pct"], START_YEAR, END_YEAR)
print(test_df.shape)
display(test_df.head())


## 4. Download all indicators

We now create a raw dataset for each indicator. Keeping the raw extracts separate is useful for auditability.


In [ ]:
# 4. DOWNLOAD ALL INDICATORS

raw_data = {}

for name, code_value in INDICATORS.items():
    print(f"Downloading {name} ({code_value})...")
    raw_data[name] = fetch_world_bank_indicator(
        code_value,
        start_year=START_YEAR,
        end_year=END_YEAR
    )
    print(f"  Rows: {len(raw_data[name]):,}")

print("\nDownload complete.")


## 5. Inspect the raw data

Before analysing anything, we inspect:
- number of rows;
- columns;
- data types;
- missing values;
- year coverage.

This is the beginning of our data-quality audit.


In [ ]:
# 5. BASIC RAW DATA AUDIT

for name, df_raw in raw_data.items():
    print(f"\n===== {name} =====")
    print("Shape:", df_raw.shape)
    print("Columns:", list(df_raw.columns))
    print("Year range:", df_raw["date"].min(), "to", df_raw["date"].max())
    print("Missing value %:")
    display((df_raw.isna().mean() * 100).round(2).to_frame("missing_pct"))


## 6. Standardise each indicator

We want one analytical table where:
- each row represents a **country-year**;
- each indicator is a column;
- country ISO3 code is the main identifier.

We also remove rows without an ISO3 code because aggregates such as "World" or regions are not individual countries and should not be mixed into country-level ranking.


In [ ]:
# 6. CLEAN AND RESHAPE

cleaned_tables = []

for name, df_raw in raw_data.items():
    df = df_raw.copy()

    # Rename the value field to the business-friendly indicator name.
    df = df.rename(columns={"value": name})

    # Keep only country-level records with an ISO3 code.
    df = df[df["countryiso3code"].notna()].copy()

    # Standardise key fields.
    df["countryiso3code"] = df["countryiso3code"].astype(str).str.upper().str.strip()
    df["country_name"] = df["country"].astype(str).str.strip()
    df["year"] = pd.to_numeric(df["date"], errors="coerce").astype("Int64")

    # Keep only fields needed for joining.
    df = df[["countryiso3code", "country_name", "year", name]].copy()

    # Check duplicate country-year rows before joining.
    duplicates = df.duplicated(["countryiso3code", "year"]).sum()
    print(f"{name}: duplicate country-year rows = {duplicates}")

    cleaned_tables.append(df)

# Start with the first table and left-join the rest.
analysis_df = cleaned_tables[0].copy()

for table in cleaned_tables[1:]:
    analysis_df = analysis_df.merge(
        table,
        on=["countryiso3code", "country_name", "year"],
        how="outer",
        validate="one_to_one"
    )

analysis_df = analysis_df.sort_values(["countryiso3code", "year"]).reset_index(drop=True)

print("Combined shape:", analysis_df.shape)
display(analysis_df.head())


## 7. Data-quality audit — missingness

Missing data are not automatically errors. We quantify them and decide how to handle them.

For the portfolio, we should report:
- which indicators have missing values;
- how many countries have complete data;
- whether missingness could bias the ranking.


In [ ]:
# 7. MISSINGNESS AUDIT

indicator_columns = list(INDICATORS.keys())

missing_summary = pd.DataFrame({
    "missing_count": analysis_df[indicator_columns].isna().sum(),
    "total_rows": len(analysis_df),
})

missing_summary["missing_pct"] = (
    missing_summary["missing_count"] / missing_summary["total_rows"] * 100
).round(2)

display(missing_summary.sort_values("missing_pct", ascending=False))


## 8. Duplicate and consistency checks

A country-year should normally appear once in our analytical table.

We also check:
- impossible Internet usage percentages;
- impossible rural-population percentages;
- negative population/GDP;
- mobile subscriptions outside a plausible analytical range.

We will **flag**, not silently delete, unusual values.


In [ ]:
# 8. DUPLICATE + RANGE CHECKS

print("Duplicate country-year rows:", 
      analysis_df.duplicated(["countryiso3code", "year"]).sum())

range_checks = {
    "internet_usage_pct_outside_0_100": (
        (analysis_df["internet_usage_pct"] < 0) |
        (analysis_df["internet_usage_pct"] > 100)
    ).sum(),
    "rural_population_pct_outside_0_100": (
        (analysis_df["rural_population_pct"] < 0) |
        (analysis_df["rural_population_pct"] > 100)
    ).sum(),
    "negative_population": (analysis_df["population"] < 0).sum(),
    "negative_gdp_per_capita": (analysis_df["gdp_per_capita"] < 0).sum(),
    "negative_mobile_subscriptions": (
        analysis_df["mobile_subscriptions_per_100"] < 0
    ).sum(),
}

display(pd.Series(range_checks, name="issue_count").to_frame())


## 9. Select the latest available observation per country

A major analytical risk is comparing countries using different years.

Our first-pass approach is:
1. For each country and indicator, find the latest non-missing year.
2. Use a common reference year where possible.
3. Measure how much of the dataset is actually comparable.

We will **not pretend that all countries have 2025 observations**.


In [ ]:
# 9. LATEST AVAILABLE VALUE BY COUNTRY AND INDICATOR

def latest_by_country(df, value_column):
    temp = df[[
        "countryiso3code", "country_name", "year", value_column
    ]].dropna(subset=[value_column]).copy()

    temp = temp.sort_values(["countryiso3code", "year"])
    return temp.groupby(
        ["countryiso3code", "country_name"], as_index=False
    ).tail(1)


latest_tables = {}

for indicator in indicator_columns:
    latest_tables[indicator] = latest_by_country(
        analysis_df, indicator
    )[[
        "countryiso3code", "country_name", "year", indicator
    ]].copy()

    latest_tables[indicator] = latest_tables[indicator].rename(
        columns={"year": f"{indicator}_year"}
    )

latest_df = latest_tables["internet_usage_pct"][[
    "countryiso3code", "country_name", "internet_usage_pct",
    "internet_usage_pct_year"
]].copy()

for indicator in indicator_columns[1:]:
    latest_df = latest_df.merge(
        latest_tables[indicator],
        on=["countryiso3code", "country_name"],
        how="outer",
        validate="one_to_one"
    )

print("Latest-value country table:", latest_df.shape)
display(latest_df.head())


## 10. Comparability audit

The latest observation for one indicator may come from a different year than another indicator.

We therefore create:
- the latest year across each indicator;
- a simple year-spread measure;
- a completeness flag.

This does not solve comparability, but it makes the limitation visible.


In [ ]:
# 10. COMPARABILITY AUDIT

year_columns = [f"{x}_year" for x in indicator_columns]

latest_df["latest_year"] = latest_df[year_columns].max(axis=1)
latest_df["earliest_year"] = latest_df[year_columns].min(axis=1)
latest_df["year_spread"] = latest_df["latest_year"] - latest_df["earliest_year"]

latest_df["complete_core_data"] = latest_df[indicator_columns].notna().all(axis=1)

comparability_summary = latest_df["year_spread"].describe()
display(comparability_summary.to_frame("year_spread"))

print("Countries with complete data:",
      int(latest_df["complete_core_data"].sum()))
print("Countries in table:",
      len(latest_df))


## 11. Create a clean analytical base

For prioritisation we need a defensible comparison set.

We will keep countries with:
- Internet usage;
- population;
- GDP per capita;
- rural population;
- mobile subscriptions.

We retain the year fields so the final portfolio can disclose the observation years.


In [ ]:
# 11. ANALYTICAL BASE

required_for_analysis = [
    "internet_usage_pct",
    "population",
    "gdp_per_capita",
    "rural_population_pct",
    "mobile_subscriptions_per_100",
]

priority_df = latest_df.dropna(
    subset=required_for_analysis
).copy()

print("Countries available for core analysis:", len(priority_df))
display(priority_df.head())


## 12. Descriptive statistics

Before creating a score, understand the distributions.

We calculate:
- count;
- mean;
- median;
- minimum;
- maximum.

Because GDP per capita is highly skewed in many real-world datasets, later charts may use a logarithmic x-axis.


In [ ]:
# 12. DESCRIPTIVE STATISTICS

descriptive_stats = priority_df[required_for_analysis].describe().T
display(descriptive_stats)


## 13. Country ranking by Internet usage

This is a useful descriptive view, but it is **not yet our prioritisation framework**.

A country with low Internet usage is not automatically the highest-impact intervention opportunity.


In [ ]:
# 13. LOWEST INTERNET USAGE

lowest_usage = priority_df.sort_values(
    "internet_usage_pct", ascending=True
)[[
    "country_name",
    "countryiso3code",
    "internet_usage_pct",
    "population",
    "gdp_per_capita",
    "rural_population_pct",
    "internet_usage_pct_year"
]].head(20)

display(lowest_usage)


## 14. Build an evidence-based "gap" metric

There is no single universal GSMA target that we should invent for every country.

For this independent case study, we therefore define a **portfolio analysis benchmark** of 80% Internet usage.

This is a **project assumption**, not a claim that GSMA has an official universal 80% country target.

The gap is:

`80% benchmark - actual Internet usage`

We cap negative gaps at zero because countries already above the benchmark are not treated as having a deficit under this particular framework.


In [ ]:
# 14. PORTFOLIO BENCHMARK

USAGE_BENCHMARK = 80.0  # PROJECT ASSUMPTION, NOT AN OFFICIAL GSMA TARGET

priority_df["usage_gap_pp"] = (
    USAGE_BENCHMARK - priority_df["internet_usage_pct"]
).clip(lower=0)

priority_df["population_affected_est"] = (
    priority_df["population"] * priority_df["usage_gap_pp"] / 100
)

display(
    priority_df[[
        "country_name",
        "internet_usage_pct",
        "usage_gap_pp",
        "population",
        "population_affected_est"
    ]].sort_values(
        "population_affected_est", ascending=False
    ).head(20)
)


## 15. Why population impact matters

A percentage-point gap alone can over-prioritise small countries.

We therefore calculate an estimated number of people corresponding to the gap.

**Interpretation:** This is a simple scenario estimate, not a count of uniquely unconnected people. It assumes the benchmark gap applies proportionally to the country's population.


In [ ]:
# 15. TOP COUNTRIES BY ESTIMATED POPULATION IMPACT

top_population_impact = priority_df.sort_values(
    "population_affected_est", ascending=False
)[[
    "country_name",
    "internet_usage_pct",
    "usage_gap_pp",
    "population",
    "population_affected_est"
]].head(20)

display(top_population_impact)


## 16. Explore the relationship with GDP per capita

We test whether Internet usage is associated with GDP per capita.

This is **association analysis**, not causal inference.


In [ ]:
# 16. GDP VS INTERNET USAGE

corr_gdp = priority_df[
    ["internet_usage_pct", "gdp_per_capita"]
].corr(method="spearman").iloc[0, 1]

print(f"Spearman correlation between Internet usage and GDP per capita: {corr_gdp:.3f}")


In [ ]:
# 16B. VISUALISE GDP VS INTERNET USAGE

plot_df = priority_df[
    ["gdp_per_capita", "internet_usage_pct"]
].replace([np.inf, -np.inf], np.nan).dropna()

plt.figure(figsize=(10, 6))
plt.scatter(
    plot_df["gdp_per_capita"],
    plot_df["internet_usage_pct"],
    alpha=0.65
)
plt.xscale("log")
plt.xlabel("GDP per capita (current US$, log scale)")
plt.ylabel("Individuals using Internet (% of population)")
plt.title("Internet Usage vs GDP per Capita")
plt.grid(alpha=0.2)
plt.show()


## 17. Rural population relationship

We examine whether countries with larger rural populations tend to have lower Internet usage.

Again, this is an association and does not prove causation.


In [ ]:
# 17. RURAL POPULATION VS INTERNET USAGE

corr_rural = priority_df[
    ["internet_usage_pct", "rural_population_pct"]
].corr(method="spearman").iloc[0, 1]

print(f"Spearman correlation between Internet usage and rural population share: {corr_rural:.3f}")


In [ ]:
# 17B. VISUALISE RURAL SHARE VS INTERNET USAGE

plt.figure(figsize=(10, 6))
plt.scatter(
    priority_df["rural_population_pct"],
    priority_df["internet_usage_pct"],
    alpha=0.65
)
plt.xlabel("Rural population (% of total population)")
plt.ylabel("Individuals using Internet (% of population)")
plt.title("Internet Usage vs Rural Population Share")
plt.grid(alpha=0.2)
plt.show()


## 18. Coverage vs usage — important conceptual limitation

Mobile subscriptions are not a direct measure of mobile-broadband coverage.

We can use subscriptions as a **contextual indicator**, but we should not label them "coverage".

This distinction is important in the final BA case study.

We therefore examine the relationship without claiming it proves infrastructure availability.


In [ ]:
# 18. MOBILE SUBSCRIPTIONS VS INTERNET USAGE

corr_mobile = priority_df[
    ["internet_usage_pct", "mobile_subscriptions_per_100"]
].corr(method="spearman").iloc[0, 1]

print(
    "Spearman correlation between Internet usage and mobile subscriptions per 100 people:",
    f"{corr_mobile:.3f}"
)


## 19. Create a transparent prioritisation framework

We now translate analysis into a BA decision-support mechanism.

### Proposed dimensions

1. Usage gap — 35%
2. Population impact — 30%
3. Rural population share — 15%
4. Economic vulnerability proxy — 20%

These weights are **proposed project assumptions**. In a real organisation they would be validated with stakeholders.

### Important:
We normalise each component to a 0–100 scale using percentile ranks rather than mixing raw units such as dollars, people and percentage points.


In [ ]:
# 19. NORMALISE PRIORITISATION COMPONENTS

def percentile_score(series, higher_is_worse=True):
    ranks = series.rank(pct=True, method="average") * 100
    return ranks if higher_is_worse else 100 - ranks


priority_df["gap_score"] = percentile_score(
    priority_df["usage_gap_pp"],
    higher_is_worse=True
)

priority_df["population_impact_score"] = percentile_score(
    priority_df["population_affected_est"],
    higher_is_worse=True
)

priority_df["rural_score"] = percentile_score(
    priority_df["rural_population_pct"],
    higher_is_worse=True
)

# Lower GDP per capita = higher vulnerability score.
priority_df["economic_vulnerability_score"] = percentile_score(
    priority_df["gdp_per_capita"],
    higher_is_worse=False
)

display(priority_df[[
    "country_name",
    "gap_score",
    "population_impact_score",
    "rural_score",
    "economic_vulnerability_score"
]].head())


## 20. Calculate the Priority Score

Weights are:
- Gap: 35%
- Population impact: 30%
- Rural context: 15%
- Economic vulnerability: 20%

The score is a **decision-support tool**, not an objective measure of social need.


In [ ]:
# 20. PRIORITY SCORE

WEIGHTS = {
    "gap_score": 0.35,
    "population_impact_score": 0.30,
    "rural_score": 0.15,
    "economic_vulnerability_score": 0.20,
}

priority_df["priority_score"] = (
    priority_df["gap_score"] * WEIGHTS["gap_score"] +
    priority_df["population_impact_score"] * WEIGHTS["population_impact_score"] +
    priority_df["rural_score"] * WEIGHTS["rural_score"] +
    priority_df["economic_vulnerability_score"] * WEIGHTS["economic_vulnerability_score"]
)

priority_df["priority_band"] = pd.cut(
    priority_df["priority_score"],
    bins=[-np.inf, 33.33, 66.67, np.inf],
    labels=["Lower", "Medium", "Higher"]
)

ranking = priority_df.sort_values(
    "priority_score", ascending=False
)[[
    "country_name",
    "countryiso3code",
    "priority_score",
    "priority_band",
    "internet_usage_pct",
    "usage_gap_pp",
    "population_affected_est",
    "rural_population_pct",
    "gdp_per_capita"
]]

display(ranking.head(25))


## 21. Data confidence rating

A senior analyst should not present every score with equal confidence.

We use a simple rule based on:
- completeness of the core indicators;
- spread between the latest observation years.

This is a **proposed confidence framework**, not a statistical confidence interval.


In [ ]:
# 21. DATA CONFIDENCE

def confidence_rating(row):
    years = row[year_columns].dropna()

    completeness = row[required_for_analysis].notna().mean()

    if len(years) == 0:
        return "Low"

    spread = years.max() - years.min()

    if completeness == 1 and spread <= 1:
        return "High"
    elif completeness >= 0.8 and spread <= 3:
        return "Medium"
    else:
        return "Low"


priority_df["data_confidence"] = priority_df.apply(
    confidence_rating,
    axis=1
)

display(
    priority_df[
        ["country_name", "priority_score", "priority_band", "data_confidence"]
    ].sort_values("priority_score", ascending=False).head(25)
)


## 22. Sensitivity analysis

A useful BA question is:

> "Would the prioritisation change if stakeholders chose different weights?"

We test a second scenario:
- Gap: 25%
- Population impact: 40%
- Rural: 15%
- Economic vulnerability: 20%

This shows whether the decision is robust or highly sensitive to stakeholder preferences.


In [ ]:
# 22. SENSITIVITY ANALYSIS

ALT_WEIGHTS = {
    "gap_score": 0.25,
    "population_impact_score": 0.40,
    "rural_score": 0.15,
    "economic_vulnerability_score": 0.20,
}

priority_df["priority_score_alt"] = (
    priority_df["gap_score"] * ALT_WEIGHTS["gap_score"] +
    priority_df["population_impact_score"] * ALT_WEIGHTS["population_impact_score"] +
    priority_df["rural_score"] * ALT_WEIGHTS["rural_score"] +
    priority_df["economic_vulnerability_score"] * ALT_WEIGHTS["economic_vulnerability_score"]
)

priority_df["rank_base"] = priority_df["priority_score"].rank(
    ascending=False, method="min"
)

priority_df["rank_alt"] = priority_df["priority_score_alt"].rank(
    ascending=False, method="min"
)

priority_df["rank_change"] = priority_df["rank_base"] - priority_df["rank_alt"]

sensitivity = priority_df.sort_values(
    "priority_score", ascending=False
)[[
    "country_name",
    "priority_score",
    "priority_score_alt",
    "rank_base",
    "rank_alt",
    "rank_change"
]].head(25)

display(sensitivity)


## 23. Identify potential intervention patterns

We can create a simple diagnostic classification based on the indicators we actually have.

This is **not a causal root-cause model**. It is a screening framework for deciding where further investigation is needed.

- High gap + high rural share → rural/access investigation
- High gap + low GDP → affordability/economic barriers investigation
- High gap + relatively high mobile subscriptions → investigate adoption/usage barriers
- Otherwise → mixed/needs further investigation


In [ ]:
# 23. SIMPLE DIAGNOSTIC SEGMENTATION

gap_median = priority_df["usage_gap_pp"].median()
rural_median = priority_df["rural_population_pct"].median()
gdp_median = priority_df["gdp_per_capita"].median()
mobile_median = priority_df["mobile_subscriptions_per_100"].median()

def diagnostic_segment(row):
    high_gap = row["usage_gap_pp"] >= gap_median
    high_rural = row["rural_population_pct"] >= rural_median
    low_gdp = row["gdp_per_capita"] <= gdp_median
    high_mobile = row["mobile_subscriptions_per_100"] >= mobile_median

    if high_gap and high_rural:
        return "Investigate rural/access barriers"
    elif high_gap and low_gdp:
        return "Investigate affordability/economic barriers"
    elif high_gap and high_mobile:
        return "Investigate usage/adoption barriers"
    elif high_gap:
        return "Mixed barriers — investigate"
    else:
        return "Lower immediate gap"

priority_df["diagnostic_segment"] = priority_df.apply(
    diagnostic_segment,
    axis=1
)

display(
    priority_df[
        [
            "country_name",
            "internet_usage_pct",
            "usage_gap_pp",
            "diagnostic_segment"
        ]
    ].sort_values("usage_gap_pp", ascending=False).head(25)
)


## 24. Visualise the prioritisation

This chart is designed as a portfolio-ready decision view:
- x-axis = Internet usage;
- y-axis = estimated population affected by the benchmark gap;
- point size = population;
- labels = selected high-priority countries.

The exact top countries are generated from the live data when the notebook runs.


In [ ]:
# 24. PRIORITY VISUAL

chart_df = priority_df.sort_values(
    "priority_score", ascending=False
).head(20).copy()

plt.figure(figsize=(12, 8))

plt.scatter(
    chart_df["internet_usage_pct"],
    chart_df["population_affected_est"],
    s=np.sqrt(chart_df["population"]) / 20 + 20,
    alpha=0.65
)

for _, row in chart_df.head(10).iterrows():
    plt.annotate(
        row["country_name"],
        (row["internet_usage_pct"], row["population_affected_est"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8
    )

plt.axvline(
    USAGE_BENCHMARK,
    linestyle="--",
    linewidth=1,
    label=f"Portfolio benchmark: {USAGE_BENCHMARK:.0f}%"
)

plt.xlabel("Internet usage (% of population)")
plt.ylabel("Estimated population corresponding to usage gap")
plt.title("Connectivity Gap vs Estimated Population Impact")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 25. Visualise the top priority markets

This is a simple management view of the proposed ranking.


In [ ]:
# 25. TOP PRIORITIES BAR CHART

top10 = priority_df.sort_values(
    "priority_score", ascending=False
).head(10).sort_values("priority_score")

plt.figure(figsize=(10, 6))
plt.barh(
    top10["country_name"],
    top10["priority_score"]
)
plt.xlabel("Priority score (0–100)")
plt.ylabel("Country")
plt.title("Top 10 Markets by Proposed Connectivity Priority Score")
plt.grid(axis="x", alpha=0.2)
plt.show()


## 26. Create a stakeholder-ready priority table

This table is more useful than giving stakeholders raw technical output.


In [ ]:
# 26. STAKEHOLDER-READY OUTPUT

stakeholder_table = priority_df.sort_values(
    "priority_score", ascending=False
)[[
    "country_name",
    "countryiso3code",
    "priority_score",
    "priority_band",
    "data_confidence",
    "internet_usage_pct",
    "internet_usage_pct_year",
    "usage_gap_pp",
    "population",
    "population_affected_est",
    "rural_population_pct",
    "gdp_per_capita",
    "mobile_subscriptions_per_100",
    "diagnostic_segment"
]].copy()

display(stakeholder_table.head(20))


## 27. BA Requirements

The following are **proposed requirements derived from the case-study problem**, not requirements elicited from real GSMA employees.

They demonstrate how an analyst would translate the problem into a solution specification.


In [ ]:
# 27. PROPOSED BA REQUIREMENTS

requirements = pd.DataFrame([
    {
        "ID": "BR-001",
        "Type": "Business",
        "Requirement": "Provide a consistent method for prioritising connectivity markets.",
        "Priority": "Must",
        "Rationale": "Reduce inconsistent/ad-hoc prioritisation."
    },
    {
        "ID": "BR-002",
        "Type": "Business",
        "Requirement": "Identify the population impact associated with connectivity gaps.",
        "Priority": "Must",
        "Rationale": "Avoid ranking solely on percentages."
    },
    {
        "ID": "BR-003",
        "Type": "Business",
        "Requirement": "Provide contextual indicators that support root-cause investigation.",
        "Priority": "Should",
        "Rationale": "Different gaps may require different interventions."
    },
    {
        "ID": "FR-001",
        "Type": "Functional",
        "Requirement": "Allow analysts to filter and rank countries by priority score.",
        "Priority": "Must",
        "Rationale": "Support decision-making."
    },
    {
        "ID": "FR-002",
        "Type": "Functional",
        "Requirement": "Display the source year for each key indicator.",
        "Priority": "Must",
        "Rationale": "Make data freshness visible."
    },
    {
        "ID": "FR-003",
        "Type": "Functional",
        "Requirement": "Display a data-confidence rating.",
        "Priority": "Should",
        "Rationale": "Prevent overconfidence in incomplete data."
    },
    {
        "ID": "NFR-001",
        "Type": "Non-functional",
        "Requirement": "Definitions and calculation rules shall be documented.",
        "Priority": "Must",
        "Rationale": "Support reproducibility and governance."
    },
])

display(requirements)


## 28. User Stories and Acceptance Criteria

Again, these are proposed portfolio artefacts.

The goal is to demonstrate that the BA can move from a business problem to testable requirements.


In [ ]:
# 28. USER STORIES

user_stories = pd.DataFrame([
    {
        "ID": "US-001",
        "User Story": "As a programme analyst, I want to rank markets by connectivity priority so that I can focus investigation on the highest-impact markets.",
        "Acceptance Criteria": "Given valid country data, when the ranking is generated, then countries are ordered by the defined priority-score logic."
    },
    {
        "ID": "US-002",
        "User Story": "As a programme manager, I want to see the population impact of a connectivity gap so that I can understand potential scale.",
        "Acceptance Criteria": "The output displays the benchmark gap and estimated population impact for each country."
    },
    {
        "ID": "US-003",
        "User Story": "As an analyst, I want to see data freshness and confidence so that I can assess whether a result is reliable enough for decision-making.",
        "Acceptance Criteria": "Each country displays indicator years and a data-confidence category."
    },
    {
        "ID": "US-004",
        "User Story": "As a stakeholder, I want to understand possible barrier patterns so that I can determine what further investigation is needed.",
        "Acceptance Criteria": "Countries with material gaps are assigned a transparent diagnostic segment based on documented rules."
    }
])

display(user_stories)


## 29. As-Is and To-Be process

Because we do not have access to an internal GSMA workflow, this is a **hypothetical process model** for the independent case study.

### Hypothetical As-Is
Request → analyst searches multiple sources → manual spreadsheet compilation → meeting → decision.

### Proposed To-Be
Scheduled data refresh → automated QA → standard scoring → analyst validation → stakeholder review → prioritisation decision → outcome tracking.


In [ ]:
# 29. PROCESS COMPARISON TABLE

process_comparison = pd.DataFrame([
    {
        "Stage": 1,
        "As-Is (hypothetical)": "Programme team requests connectivity analysis",
        "Pain Point": "Request-driven and inconsistent",
        "To-Be (proposed)": "Standard quarterly review cycle",
        "Benefit": "Predictable governance"
    },
    {
        "Stage": 2,
        "As-Is (hypothetical)": "Analyst searches multiple reports/spreadsheets",
        "Pain Point": "Manual effort and inconsistent sources",
        "To-Be (proposed)": "Controlled data-source catalogue and refresh",
        "Benefit": "Consistency and traceability"
    },
    {
        "Stage": 3,
        "As-Is (hypothetical)": "Manual country comparison",
        "Pain Point": "Different assumptions/calculations",
        "To-Be (proposed)": "Standard prioritisation framework",
        "Benefit": "Repeatable decisions"
    },
    {
        "Stage": 4,
        "As-Is (hypothetical)": "Stakeholder meeting",
        "Pain Point": "Limited evidence trail",
        "To-Be (proposed)": "Review dashboard + documented assumptions",
        "Benefit": "Transparent challenge process"
    },
    {
        "Stage": 5,
        "As-Is (hypothetical)": "Decision stored in presentation/spreadsheet",
        "Pain Point": "Weak audit trail",
        "To-Be (proposed)": "Decision + rationale + owner + outcome tracking",
        "Benefit": "Accountability and learning"
    }
])

display(process_comparison)


## 30. UAT Test Cases

These are proposed tests for a future dashboard/decision tool.


In [ ]:
# 30. UAT TEST CASES

uat_cases = pd.DataFrame([
    {
        "Test ID": "UAT-001",
        "Requirement": "FR-001",
        "Scenario": "Rank countries by priority score",
        "Expected Result": "Countries appear in descending priority-score order.",
        "Priority": "High"
    },
    {
        "Test ID": "UAT-002",
        "Requirement": "FR-002",
        "Scenario": "Inspect indicator year",
        "Expected Result": "The source year for each indicator is visible.",
        "Priority": "High"
    },
    {
        "Test ID": "UAT-003",
        "Requirement": "FR-003",
        "Scenario": "Inspect data confidence",
        "Expected Result": "Each country has a documented confidence category.",
        "Priority": "Medium"
    },
    {
        "Test ID": "UAT-004",
        "Requirement": "BR-002",
        "Scenario": "Review population impact",
        "Expected Result": "Gap and estimated population impact are calculated consistently.",
        "Priority": "High"
    },
    {
        "Test ID": "UAT-005",
        "Requirement": "NFR-001",
        "Scenario": "Review methodology",
        "Expected Result": "Indicator definitions, benchmark, weights and assumptions are documented.",
        "Priority": "High"
    }
])

display(uat_cases)


## 31. Requirements Traceability Matrix (RTM)

This links the proposed business requirements to user stories and UAT tests.


In [ ]:
# 31. RTM

rtm = pd.DataFrame([
    {"Requirement": "BR-001", "User Story": "US-001", "UAT": "UAT-001", "Status": "Covered"},
    {"Requirement": "BR-002", "User Story": "US-002", "UAT": "UAT-004", "Status": "Covered"},
    {"Requirement": "BR-003", "User Story": "US-004", "UAT": "UAT-005", "Status": "Covered"},
    {"Requirement": "FR-001", "User Story": "US-001", "UAT": "UAT-001", "Status": "Covered"},
    {"Requirement": "FR-002", "User Story": "US-003", "UAT": "UAT-002", "Status": "Covered"},
    {"Requirement": "FR-003", "User Story": "US-003", "UAT": "UAT-003", "Status": "Covered"},
    {"Requirement": "NFR-001", "User Story": "US-003", "UAT": "UAT-005", "Status": "Covered"},
])

display(rtm)


## 32. Data Dictionary

A portfolio-quality BA project should define what each field means and where it came from.


In [ ]:
# 32. DATA DICTIONARY

data_dictionary = pd.DataFrame([
    {
        "Field": "countryiso3code",
        "Definition": "ISO-style three-letter country code supplied by World Bank API.",
        "Source": "World Bank",
        "Type": "Text",
        "Business Use": "Country identifier"
    },
    {
        "Field": "internet_usage_pct",
        "Definition": "Individuals using the Internet as a percentage of population.",
        "Source": "World Bank / ITU indicator IT.NET.USER.ZS",
        "Type": "Numeric %",
        "Business Use": "Connectivity adoption"
    },
    {
        "Field": "mobile_subscriptions_per_100",
        "Definition": "Mobile cellular subscriptions per 100 people.",
        "Source": "World Bank / ITU indicator IT.CEL.SETS.P2",
        "Type": "Numeric",
        "Business Use": "Mobile-market context"
    },
    {
        "Field": "population",
        "Definition": "Total population.",
        "Source": "World Bank indicator SP.POP.TOTL",
        "Type": "Numeric",
        "Business Use": "Scale / impact"
    },
    {
        "Field": "gdp_per_capita",
        "Definition": "GDP per capita in current US dollars.",
        "Source": "World Bank indicator NY.GDP.PCAP.CD",
        "Type": "Numeric",
        "Business Use": "Economic context"
    },
    {
        "Field": "rural_population_pct",
        "Definition": "Rural population as percentage of total population.",
        "Source": "World Bank indicator SP.RUR.TOTL.ZS",
        "Type": "Numeric %",
        "Business Use": "Rural context"
    },
])

display(data_dictionary)


## 33. Executive summary generator

This cell creates a small factual summary from the actual results in the notebook. It does not invent country names or numbers.


In [ ]:
# 33. EXECUTIVE SUMMARY METRICS

highest_priority = priority_df.sort_values(
    "priority_score", ascending=False
).head(5)

median_usage = priority_df["internet_usage_pct"].median()
median_gap = priority_df["usage_gap_pp"].median()
total_estimated_impact = priority_df["population_affected_est"].sum()

print("EXECUTIVE SUMMARY")
print("-" * 60)
print(f"Countries in core analysis: {len(priority_df):,}")
print(f"Median Internet usage: {median_usage:.1f}%")
print(f"Median benchmark gap: {median_gap:.1f} percentage points")
print(f"Estimated population represented by benchmark gaps: {total_estimated_impact:,.0f}")
print("\nTop 5 proposed priority markets:")
for i, (_, row) in enumerate(highest_priority.iterrows(), start=1):
    print(
        f"{i}. {row['country_name']} | "
        f"Score {row['priority_score']:.1f} | "
        f"Confidence {row['data_confidence']}"
    )


## 34. Export portfolio outputs

The final files can be downloaded from Colab after this cell runs.

We export:
- clean analytical dataset;
- stakeholder priority table;
- requirements;
- user stories;
- UAT cases;
- RTM;
- data dictionary;
- process comparison.

These can later support a Power BI dashboard and portfolio case study.


In [ ]:
# 34. EXPORT OUTPUTS

output_dir = Path("project4_outputs")
output_dir.mkdir(exist_ok=True)

priority_df.to_csv(
    output_dir / "connectivity_priority_dataset.csv",
    index=False
)

stakeholder_table.to_csv(
    output_dir / "stakeholder_priority_table.csv",
    index=False
)

requirements.to_csv(
    output_dir / "ba_requirements.csv",
    index=False
)

user_stories.to_csv(
    output_dir / "user_stories.csv",
    index=False
)

uat_cases.to_csv(
    output_dir / "uat_test_cases.csv",
    index=False
)

rtm.to_csv(
    output_dir / "requirements_traceability_matrix.csv",
    index=False
)

data_dictionary.to_csv(
    output_dir / "data_dictionary.csv",
    index=False
)

process_comparison.to_csv(
    output_dir / "as_is_to_be_process.csv",
    index=False
)

print("Files created:")
for path in sorted(output_dir.iterdir()):
    print(" -", path)


## 35. Final methodology checklist

Before presenting this project in a portfolio, verify:

### Evidence
- [ ] Public sources are cited.
- [ ] Data retrieval date is recorded.
- [ ] Indicator definitions are documented.
- [ ] Observation years are visible.

### Data quality
- [ ] Missingness was measured.
- [ ] Duplicates were checked.
- [ ] Range checks were performed.
- [ ] Comparability limitations were disclosed.

### Analysis
- [ ] Internet usage is distinguished from mobile subscriptions.
- [ ] Association is not described as causation.
- [ ] Population impact is explained as an estimate.
- [ ] The 80% benchmark is labelled as a project assumption.

### BA
- [ ] Stakeholders are clearly labelled as proposed/simulated unless actually interviewed.
- [ ] As-Is process is labelled hypothetical.
- [ ] To-Be process is labelled proposed.
- [ ] Requirements are traceable to UAT.
- [ ] Solution recommendations follow the evidence.

### Portfolio
- [ ] Do not claim you worked for GSMA.
- [ ] Do not claim the hypothetical process is GSMA's actual process.
- [ ] Explain what you personally built and analysed.


# End of Project 4 Notebook

### Recommended next stage

Use the exported `connectivity_priority_dataset.csv` to build a **Power BI prototype** with:
1. Executive KPI cards
2. Country priority ranking
3. Connectivity gap map
4. Population-impact view
5. Diagnostic segment view
6. Data-confidence indicator
7. Country drill-down

The BA case study should then connect the dashboard back to the proposed **As-Is → To-Be process, requirements, UAT and change-impact plan**.
